# Thinker-Talker Projection Layer Training

Trains the projection layer of the Thinker-Talker audio extension: a small MLP
that maps a frozen thinker LLM's last-layer hidden states into a frozen
talker's input embedding space. The trained weights are exported as GGUF for
audio inference in `cluster/applications/rust_de_llama`
(see `cluster/applications/rust_de_llama/README.md`).

```
Thinker LLM (frozen) -> hidden states -> fc1 -> SiLU -> fc2 -> talker embeddings
                                                                     |
        speaker prompt (voice conditioning) ---------------------> talker (frozen)
                                                                     |
                                                          audio codes (autoregressive)
```

The projection stands in for the talker's own text-token embeddings: at
inference the runtime prefills the talker with the speaker scaffolding, the
projected hidden states in place of the text, and lets the talker generate
audio codes until end-of-generation. Training mirrors that exactly, supervising
the talker's continuation with teacher forcing.

Both LLMs are frozen; only `fc1`/`fc2` are trained. `output_dim` is therefore
fixed by the talker's hidden size, and `input_dim` by the thinker's -- the
runtime validates both at load.

## Where the targets come from

The talker is pretrained and already speaks: given text as *its own* tokens it
emits an intelligible continuation, which the runtime's talker-native path uses
and `test_generate_native_speech` verifies. That path is this notebook's
teacher. For each corpus sentence the talker reads the normalized text and its
continuation is recorded verbatim; the projection is then trained to make the
same talker emit that same continuation from the thinker's hidden states
instead. The supervision is therefore real speech in the speaker prompt's own
voice, in the talker's own format, and needs no external recordings and no
WavTokenizer encoder.

Two consequences are worth being explicit about:

- The thinker reads the **raw** sentence while the targets come from the
  normalized one. Normalization keeps only letters, so this is what lets the
  projected path carry case and punctuation that the talker-native path cannot
  see at all.
- Distillation cannot teach the talker to say what it could not already say.
  The corpus is digit-free English, so numbers and other languages are outside
  the trained distribution; reaching them needs targets from real recordings,
  which is a separate data problem.

Everything is cached: `data/train.json` is only regenerated when absent, and
training resumes from `data/checkpoint.pt`. The notebook selects CUDA when it
is available and falls back to CPU otherwise.

## 1. Import Dependencies

In [ ]:
import collections
import dataclasses
import json
import pathlib
import struct

import huggingface_hub
import numpy
import torch
import torch.nn
import torch.nn.functional
import torch.utils.data
import tqdm.auto
import transformers

## 2. Configuration and Constants

In [ ]:
THINKER_MODEL = "HuggingFaceTB/SmolLM2-135M-Instruct"
TALKER_MODEL = "OuteAI/OuteTTS-0.2-500M"
HIDDEN_DIM = 2048

DATA_DIRECTORY = pathlib.Path("data")
DATA_PATH = DATA_DIRECTORY / "train.json"
CHECKPOINT_PATH = DATA_DIRECTORY / "checkpoint.pt"
OUTPUT_PATH = pathlib.Path("thinker-talker-projection.gguf")

# The talker encodes the decoder's 4096 codes as the contiguous token run
# <|0|>..<|4095|>; the runtime resolves the same range by tokenization.
AUDIO_CODE_COUNT = 4096
AUDIO_CODE_FIRST_MARKER = "<|0|>"

# Distillation corpus: simple, clean, digit-free English prose. Short
# sentences keep every continuation well inside one talker run.
CORPUS_REPOSITORY = "roneneldan/TinyStories"
CORPUS_FILE = "TinyStories-valid.txt"
CORPUS_SIZE = 1200
MINIMUM_WORDS = 4
MAXIMUM_WORDS = 16
# The talker spends ~30 tokens on a word (its text, a duration, and ~25 codes
# at 75 codes/second), so this clears MAXIMUM_WORDS with room to spare. A
# continuation that reaches it stopped early and is dropped rather than kept
# truncated.
MAXIMUM_NEW_TOKENS = 640
GENERATION_BATCH_SIZE = 4

# Sentences to train on, out of the distilled corpus, and how many times each is
# seen. Separate from CORPUS_SIZE because distillation is cached: the corpus is
# generated once, and these decide how a run spends its budget on it.
#
# These numbers are not the thing to tune, and section 11's diagnostic is why.
# Every run so far, judged free-running on a *training* sentence -- so all of
# these are underfitting rather than a generalization gap:
#
#   corpus  exposures  loss   free-running output
#   ------  ---------  -----  --------------------------------------------------
#   (none)          0   2.66  the talker's own prior; the projection is ignored
#        8        100   0.25  every word, and it stops on its own
#     1200          6   1.22  fluent but the wrong sentence -- the corpus's
#                             marginal distribution, not its conditional
#      150         40   0.68  the opening words exact, then it loses the thread
#      150         80   0.39  WORSE than at 40, at half the loss
#
# The 8-sentence run is what proves the architecture carries text at all. The
# last row is the problem: halving the loss made free running worse, so over
# this range the loss is anti-correlated with the goal, and more epochs buy
# less of what is wanted. The cause is in the objective, not the schedule --
# see section 10.
TRAINING_SAMPLES = 150
EPOCHS = 40
RANDOM_STATE = 42
BATCH_SIZE = 2
LEARNING_RATE = 1e-3

torch.manual_seed(RANDOM_STATE)
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
# Frozen models only supply a differentiable loss, so bf16 buys generation and
# training throughput without touching the projection's own fp32 weights. The
# CPU path needs avx512_bf16 to be worth it; fp32 is the safe fallback.
FROZEN_DTYPE = (
    torch.float16
    if DEVICE.type == "cuda"
    else torch.bfloat16
    if torch.cpu._is_avx512_bf16_supported()
    else torch.float32
)
DATA_DIRECTORY.mkdir(exist_ok=True)
print(
    f"Device: {DEVICE}, frozen models: {FROZEN_DTYPE}, threads: {torch.get_num_threads()}"
)

## 3. Projection Layer

MLP adapter (`fc1` -> SiLU -> `fc2`); the tensor names are the contract with
the GGUF consumer.

In [ ]:
@dataclasses.dataclass
class ProjectionConfig:
    input_dim: int
    hidden_dim: int
    output_dim: int


class ProjectionLayer(torch.nn.Module):
    """MLP projection layer: LLM hidden states -> audio token logits."""

    def __init__(self, config: ProjectionConfig):
        super().__init__()
        self.config = config
        self.fc1 = torch.nn.Linear(config.input_dim, config.hidden_dim)
        self.fc2 = torch.nn.Linear(config.hidden_dim, config.output_dim)

    def forward(self, hidden_states: torch.Tensor) -> torch.Tensor:
        x = self.fc1(hidden_states)
        x = torch.nn.functional.silu(x)
        x = self.fc2(x)
        return x

## 4. Text Normalization

The teacher reads text through the talker's own tokenizer, which needs the
normalized form the runtime's talker-native path builds: words lower-cased,
stripped to letters and separated by `<|text_sep|>`. This mirrors
`normalize_text` in `src/audio/talker.rs`, which in turn ports llama.cpp
`tools/tts/tts.cpp` `process_text` -- the targets are only faithful if the two
agree, so the Rust unit tests' own vectors are asserted here.

Upstream also spells digit runs out ("42" becomes "forty-two") before dropping
non-letters. That branch is not ported: the corpus filter rejects digits
outright, which keeps this function exactly equivalent on everything it is
ever handed and keeps the number spelling in one language rather than two.

In [ ]:
# Characters normalize_text turns into a word break rather than dropping.
WORD_BREAK_CHARACTERS = "-_/,.\\"


def normalize_text(text: str) -> str:
    """Mirrors normalize_text in src/audio/talker.rs for digit-free input."""
    assert not any(character.isdigit() for character in text), (
        f"number spelling is not ported; the corpus filter must reject {text!r}"
    )
    lowered = "".join(
        " "
        if character in WORD_BREAK_CHARACTERS
        else character
        if (character.isascii() and character.isalpha()) or character.isspace()
        else ""
        for character in text.lower()
    )
    return "".join(f"{word}<|text_sep|>" for word in lowered.split())


# Vectors taken from the Rust unit tests, which are the contract this mirrors.
assert normalize_text("Hello, world!") == "hello<|text_sep|>world<|text_sep|>"
assert normalize_text("!!") == ""
assert (
    normalize_text("well-known ok") == "well<|text_sep|>known<|text_sep|>ok<|text_sep|>"
)
print("normalize_text agrees with the Rust vectors")

## 5. Load Thinker and Talker

Both are frozen: the thinker supplies hidden states, the talker both writes the
training targets (reading text as its own tokens) and scores them (reading the
projection's output). Only the projection trains.

In [ ]:
print(f"Loading thinker: {THINKER_MODEL}")

tokenizer = transformers.AutoTokenizer.from_pretrained(THINKER_MODEL)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

model = transformers.AutoModelForCausalLM.from_pretrained(
    THINKER_MODEL,
    dtype=FROZEN_DTYPE,
    device_map="auto" if DEVICE.type == "cuda" else None,
)
if DEVICE.type != "cuda":
    model.to(DEVICE)
model.eval()
model.requires_grad_(False)

INPUT_DIM = model.config.hidden_size
print(f"Thinker hidden dimension: {INPUT_DIM}")

print(f"Loading talker: {TALKER_MODEL}")

talker_tokenizer = transformers.AutoTokenizer.from_pretrained(TALKER_MODEL)
if talker_tokenizer.pad_token is None:
    talker_tokenizer.pad_token = talker_tokenizer.eos_token
# Batched generation pads on the left so every row's continuation starts at the
# same column; nothing else reads the talker tokenizer's padding.
talker_tokenizer.padding_side = "left"

talker = transformers.AutoModelForCausalLM.from_pretrained(
    TALKER_MODEL,
    dtype=FROZEN_DTYPE,
    device_map="auto" if DEVICE.type == "cuda" else None,
)
if DEVICE.type != "cuda":
    talker.to(DEVICE)
talker.eval()
talker.requires_grad_(False)

OUTPUT_DIM = talker.config.hidden_size
AUDIO_CODE_FIRST_TOKEN = talker_tokenizer.convert_tokens_to_ids(AUDIO_CODE_FIRST_MARKER)
assert isinstance(AUDIO_CODE_FIRST_TOKEN, int), (
    "talker does not define <|0|> as one token"
)
assert (
    talker_tokenizer.convert_tokens_to_ids(f"<|{AUDIO_CODE_COUNT - 1}|>")
    - AUDIO_CODE_FIRST_TOKEN
    == AUDIO_CODE_COUNT - 1
), "talker does not encode audio codes contiguously"

# The token the talker stops on. It terminates every recorded continuation, so
# the projection is supervised to reproduce the stop as well as the speech --
# without it the projected path would run to the runtime's code cap every time.
END_OF_GENERATION_TOKEN = talker.config.eos_token_id
assert isinstance(END_OF_GENERATION_TOKEN, int), "talker has no single eos token"
print(
    f"Talker hidden dimension: {OUTPUT_DIM}, '<|0|>' = {AUDIO_CODE_FIRST_TOKEN}, "
    f"eos = {END_OF_GENERATION_TOKEN} "
    f"({talker_tokenizer.convert_ids_to_tokens(END_OF_GENERATION_TOKEN)})"
)

## 6. Speaker Prompt

The speaker prompt is the in-context voice example the talker continues from,
and the same asset the runtime embeds -- it is read from the Rust tree rather
than copied, so the two cannot drift. It surrounds the text on both sides, so
its halves are tokenized once and reused for every sentence and every batch.

In [ ]:
RUST_TALKER_ASSETS = pathlib.Path(
    "../cluster/applications/rust_de_llama/src/audio/talker"
)
SPEAKER_TEXT = (RUST_TALKER_ASSETS / "speaker_text.txt").read_text()
SPEAKER_AUDIO = (RUST_TALKER_ASSETS / "speaker_audio.txt").read_text()


def talker_tokens(text: str) -> list[int]:
    return talker_tokenizer(text, add_special_tokens=False)["input_ids"]


PROMPT_PREFIX_TOKENS = talker_tokens(f"<|im_start|>\n{SPEAKER_TEXT}")
PROMPT_SUFFIX_TOKENS = talker_tokens(f"<|text_end|>\n{SPEAKER_AUDIO}")
print(
    f"Speaker prompt: {len(PROMPT_PREFIX_TOKENS)} + [text] + "
    f"{len(PROMPT_SUFFIX_TOKENS)} tokens"
)

## 7. Distillation Corpus

Sentences for the teacher to read. The filter is what keeps the targets honest
rather than what makes them diverse:

- **ASCII, digit-free** -- `normalize_text` above is only equivalent to the
  Rust one on this subset, and the talker's normalization can say nothing else.
- **Short** -- every continuation must finish inside one talker run, so a
  target is never a truncated utterance.
- **Normalization-stable** -- a sentence whose word count changes under
  normalization would be spoken differently from what the thinker reads, which
  is exactly the misalignment the projection would then have to absorb.

In [ ]:
SENTENCE_END_CHARACTERS = ".!?"


def split_sentences(text: str) -> list[str]:
    sentences = []
    start = 0
    for index, character in enumerate(text):
        if character in SENTENCE_END_CHARACTERS or character == "\n":
            sentence = text[start : index + 1].strip()
            if sentence:
                sentences.append(sentence)
            start = index + 1
    return sentences


def acceptable(sentence: str) -> bool:
    if not sentence.isascii() or any(character.isdigit() for character in sentence):
        return False
    if not sentence.endswith(tuple(SENTENCE_END_CHARACTERS)):
        return False
    words = sentence.split()
    if not MINIMUM_WORDS <= len(words) <= MAXIMUM_WORDS:
        return False
    normalized = normalize_text(sentence)
    return normalized.count("<|text_sep|>") == len(words)


def build_corpus(size: int) -> list[str]:
    corpus_file = huggingface_hub.hf_hub_download(
        repo_id=CORPUS_REPOSITORY, filename=CORPUS_FILE, repo_type="dataset"
    )
    text = pathlib.Path(corpus_file).read_text(encoding="utf-8")
    text = text.replace("<|endoftext|>", "\n")

    seen = set()
    corpus = []
    for sentence in split_sentences(text):
        if not acceptable(sentence) or sentence in seen:
            continue
        seen.add(sentence)
        corpus.append(sentence)
        if len(corpus) == size:
            break
    return corpus


corpus = build_corpus(CORPUS_SIZE)
assert len(corpus) == CORPUS_SIZE, f"corpus is short: {len(corpus)}"
word_counts = [len(sentence.split()) for sentence in corpus]
print(f"Corpus: {len(corpus)} sentences, {sum(word_counts)} words")
print(f"  words per sentence: {min(word_counts)}-{max(word_counts)}")
for sentence in corpus[:3]:
    print(f"  {sentence!r}")
    print(f"    -> {normalize_text(sentence)}")

## 8. Generate Training Targets

The teacher pass. Each sentence is read by the talker as its own tokens --
`[speaker prefix | normalized text | speaker suffix]`, exactly the runtime's
talker-native prompt -- and the continuation it generates is recorded verbatim:

```
\nthe<|t_0.52|><|code_start|><|533|><|1218|>...<|code_end|>\nquick<|t_0.40|>...<|im_end|>
```

Note what is in there besides codes: the talker writes each **word** out before
the codes that speak it. Keeping those tokens in the target is what grounds the
projection in content -- on the projected path the talker never sees the text,
so emitting the right word is something it can only get from the projection.
The runtime's `token_to_code` drops them again when it collects codes, so the
two ends agree.

Greedy decoding makes this reproducible and matches the sampler the Rust test
drives (`temperature = 0.0`). Sentences are grouped by length so a batch is not
held open by its longest row, and any continuation that fails to stop on its
own is dropped rather than kept truncated.

Written to `data/train.json` and reused on the next run.

In [ ]:
def native_prompt(sentence: str) -> list[int]:
    """The runtime's talker-native prompt for one sentence."""
    return (
        PROMPT_PREFIX_TOKENS
        + talker_tokens(normalize_text(sentence))
        + PROMPT_SUFFIX_TOKENS
    )


def generate_targets(sentences: list[str]) -> list[dict]:
    """Record the talker's own continuation for each sentence."""
    # Longest first: a batch runs until its slowest row stops, so mixing
    # lengths would spend the short rows' steps generating padding.
    order = sorted(range(len(sentences)), key=lambda i: -len(sentences[i].split()))
    samples = []
    truncated = 0

    progress = tqdm.auto.tqdm(
        range(0, len(order), GENERATION_BATCH_SIZE), desc="Distilling"
    )
    for start in progress:
        indices = order[start : start + GENERATION_BATCH_SIZE]
        prompts = [native_prompt(sentences[i]) for i in indices]
        width = max(len(prompt) for prompt in prompts)

        input_ids = torch.full(
            (len(prompts), width), talker_tokenizer.pad_token_id, dtype=torch.long
        )
        attention_mask = torch.zeros((len(prompts), width), dtype=torch.long)
        for row, prompt in enumerate(prompts):
            input_ids[row, width - len(prompt) :] = torch.tensor(prompt)
            attention_mask[row, width - len(prompt) :] = 1

        with torch.no_grad():
            generated = talker.generate(
                input_ids=input_ids.to(DEVICE),
                attention_mask=attention_mask.to(DEVICE),
                max_new_tokens=MAXIMUM_NEW_TOKENS,
                do_sample=False,
                eos_token_id=END_OF_GENERATION_TOKEN,
                pad_token_id=talker_tokenizer.pad_token_id,
            )

        for row, index in enumerate(indices):
            continuation = generated[row, width:].tolist()
            if END_OF_GENERATION_TOKEN not in continuation:
                truncated += 1
                continue
            # Keep the stop token itself: the projection has to learn to end.
            end = continuation.index(END_OF_GENERATION_TOKEN) + 1
            samples.append(
                {"text": sentences[index], "audio_tokens": continuation[:end]}
            )
        progress.set_postfix(kept=len(samples), dropped=truncated)

    print(f"Distilled {len(samples)} continuations, dropped {truncated} unfinished")
    return samples


if DATA_PATH.exists():
    samples = json.loads(DATA_PATH.read_text())
    print(f"Reusing {DATA_PATH}: {len(samples)} samples")
else:
    samples = generate_targets(corpus)
    DATA_PATH.write_text(json.dumps(samples))
    print(f"Wrote {DATA_PATH}: {len(samples)} samples")

assert samples, "no training samples"
code_counts = [
    sum(
        1
        for token in sample["audio_tokens"]
        if AUDIO_CODE_FIRST_TOKEN <= token < AUDIO_CODE_FIRST_TOKEN + AUDIO_CODE_COUNT
    )
    for sample in samples
]
lengths = [len(sample["audio_tokens"]) for sample in samples]
print(f"Continuation tokens: {min(lengths)}-{max(lengths)}")
print(
    f"Audio codes: {min(code_counts)}-{max(code_counts)} "
    f"({sum(code_counts) / len(code_counts) / 75:.1f}s of speech on average)"
)
assert all(
    sample["audio_tokens"][-1] == END_OF_GENERATION_TOKEN for sample in samples
), "a target does not end on the stop token"

## 9. Dataset

The thinker reads the raw sentence; the talker's recorded continuation is the
target.

Two kinds of padding could appear here and only one of them is harmless. The
**continuation** is padded on the right to the batch's longest: those positions
are last, nothing is generated after them, and `-100` keeps them out of the
loss. The **text** cannot be padded at all -- it sits between the two halves of
the speaker prompt, so a padded position is a dead embedding exactly where the
talker expects a word, and every position after it shifts. Batches are
therefore built from sentences whose thinker tokenization has the same length,
which is what `LengthBucketSampler` does. It costs nothing but a shuffle: the
corpus has many sentences per length.

In [ ]:
class TextAudioDataset(torch.utils.data.Dataset):
    """Raw sentence -> thinker tokens, plus the talker's recorded continuation.

    JSON format: `[{"text": "...", "audio_tokens": [...]}, ...]`, where
    `audio_tokens` is the talker's continuation for `text` -- its own token ids,
    ending on the stop token.
    """

    def __init__(self, samples: list[dict], tokenizer):
        self.samples = samples
        # No padding and no truncation: the sampler guarantees a batch shares
        # one length, and a truncated sentence would not match its target.
        self.encoded = [
            tokenizer(sample["text"], add_special_tokens=True)["input_ids"]
            for sample in samples
        ]

    def text_lengths(self) -> list[int]:
        return [len(encoding) for encoding in self.encoded]

    def __len__(self) -> int:
        return len(self.samples)

    def __getitem__(self, index: int) -> dict:
        return {
            "input_ids": torch.tensor(self.encoded[index], dtype=torch.long),
            "audio_tokens": torch.tensor(
                self.samples[index]["audio_tokens"], dtype=torch.long
            ),
        }


class LengthBucketSampler(torch.utils.data.Sampler):
    """Batches whose samples share one text length."""

    def __init__(self, lengths: list[int], batch_size: int, generator):
        self.batch_size = batch_size
        self.generator = generator
        self.buckets = collections.defaultdict(list)
        for index, length in enumerate(lengths):
            self.buckets[length].append(index)

    def __iter__(self):
        batches = []
        for indices in self.buckets.values():
            order = torch.randperm(len(indices), generator=self.generator).tolist()
            shuffled = [indices[position] for position in order]
            batches.extend(
                shuffled[start : start + self.batch_size]
                for start in range(0, len(shuffled), self.batch_size)
            )
        for position in torch.randperm(len(batches), generator=self.generator).tolist():
            yield batches[position]

    def __len__(self) -> int:
        return sum(
            -(-len(indices) // self.batch_size) for indices in self.buckets.values()
        )


def collate_fn(batch: list[dict]) -> dict:
    input_ids = torch.stack([sample["input_ids"] for sample in batch])

    # -100 masks the padded tail out of the loss; the talker's own pad id fills
    # the input side, where those positions are never read back.
    width = max(sample["audio_tokens"].size(0) for sample in batch)
    audio_tokens = torch.full((len(batch), width), -100, dtype=torch.long)
    for row, sample in enumerate(batch):
        length = sample["audio_tokens"].size(0)
        audio_tokens[row, :length] = sample["audio_tokens"]

    return {"input_ids": input_ids, "audio_tokens": audio_tokens}


training_samples = samples[:TRAINING_SAMPLES]
dataset = TextAudioDataset(training_samples, tokenizer)
generator = torch.Generator().manual_seed(RANDOM_STATE)
dataloader = torch.utils.data.DataLoader(
    dataset,
    batch_sampler=LengthBucketSampler(dataset.text_lengths(), BATCH_SIZE, generator),
    collate_fn=collate_fn,
    num_workers=0,
)

text_lengths = dataset.text_lengths()
print(
    f"Training on {len(dataset)} of {len(samples)} distilled samples, "
    f"{len(dataloader)} batches per epoch, {EPOCHS} exposures each"
)
print(f"  thinker tokens per sentence: {min(text_lengths)}-{max(text_lengths)}")

## 10. Training

Teacher-forced cross-entropy on the talker's recorded continuation, mirroring
inference. Each step assembles the talker's inputs entirely as embeddings:

```
[ speaker prefix | projected thinker hidden states | speaker suffix | continuation ]
                       (the only trainable input)                     (supervised)
```

The talker scores the continuation autoregressively, so the logits at the
position *before* each token predict it. Gradients reach only the projection --
both LLMs are frozen -- which forces the projection to encode the sentence into
whatever the talker needs to speak it, including the stop token that ends it.

`lm_head` is applied only where a token is supervised. Run over the whole
sequence it would materialize `1138 x 157696` floats per sample, about 700 MB,
and its gradient again; nothing reads the positions under the speaker prompt.
It is a smaller saving in time than in memory, but it is what leaves headroom
at `BATCH_SIZE > 1`.

Progress is checkpointed every epoch, so an interrupted run resumes instead of
starting over.

### This objective is mis-shaped, and that is the open problem

Minimizing it further makes the projection worse. Measured: 150 sentences at 40
exposures reproduce `spot saw the shiny car and said wow` free-running at loss
0.68; the same run continued to 80 exposures reaches loss **0.39** and says
`spot saw the` then `spot the next minute and shiny colors like pins thank size
seven ohm`. Half the loss, a third of the words, and it no longer stops.

The cause is above, in the shape of the target. A word costs ~30 tokens: its
text, a duration, then ~25 audio codes. Under teacher forcing the codes are
predicted with the correct word already sitting in context, so the talker's own
pretrained acoustic modelling handles them and the projection contributes
nothing -- yet they are 29 of every 30 supervised positions, so they own the
gradient. Training drives the projection toward helping with codes it is not
needed for, and the ~1 position in 30 it actually owns gets what is left. That
is why a random projection already scores ~2.7 instead of the ~12 a uniform
guess over 157k tokens would, and why loss is not evidence here.

The untried fix is to supervise what the projection is responsible for and
nothing else: mask the code positions (`<|0|>`..`<|4095|>`) to `-100` in
`collate_fn`, leaving the words, the duration markers and the stop token --
about 4 positions in 30. The codes need no supervision, because they follow
from the word, which is exactly what the talker-native path already
demonstrates.

Also needed: early stopping on section 11's diagnostic rather than on the loss,
keeping the best-by-words weights aside. The checkpoint here tracks the latest
epoch, and the 150x40 weights -- the best produced so far -- were overwritten
by the 150x80 run because the loss said the later ones were better.

In [ ]:
def extract_hidden_states(input_ids: torch.Tensor) -> torch.Tensor:
    """The thinker's last-layer hidden states, post final norm.

    This is what `hidden_states[-1]` is in transformers and what the runtime
    reads out of its llama.cpp embeddings context, so the two ends see the same
    representation.
    """
    with torch.no_grad():
        outputs = model(input_ids=input_ids, output_hidden_states=True)
    # The batch is unpadded by construction, so no attention mask is needed.
    return outputs.hidden_states[-1].float()


def talker_loss(projection: ProjectionLayer, batch: dict) -> torch.Tensor:
    """Cross-entropy of the talker's continuation given the projected text."""
    input_ids = batch["input_ids"].to(DEVICE)
    audio_tokens = batch["audio_tokens"].to(DEVICE)
    batch_size, code_width = audio_tokens.shape

    hidden_states = extract_hidden_states(input_ids)
    projected = projection(hidden_states).to(FROZEN_DTYPE)

    embed = talker.get_input_embeddings()

    def scaffolding(tokens: list[int]) -> torch.Tensor:
        return embed(
            torch.tensor(tokens, dtype=torch.long, device=DEVICE)
            .unsqueeze(0)
            .expand(batch_size, -1)
        )

    prefix_embeddings = scaffolding(PROMPT_PREFIX_TOKENS)
    suffix_embeddings = scaffolding(PROMPT_SUFFIX_TOKENS)
    # -100 marks padding, which is not a valid embedding index; the loss ignores
    # those positions, so any in-range id serves as a placeholder.
    code_inputs = audio_tokens.clone()
    code_inputs[code_inputs == -100] = talker_tokenizer.pad_token_id
    code_embeddings = embed(code_inputs)

    inputs_embeds = torch.cat(
        [prefix_embeddings, projected, suffix_embeddings, code_embeddings], dim=1
    )
    states = talker.model(inputs_embeds=inputs_embeds).last_hidden_state

    # The continuation's first token is predicted by the last suffix position,
    # so the aligned states start one position earlier.
    code_start = (
        prefix_embeddings.size(1) + projected.size(1) + suffix_embeddings.size(1)
    )
    predicted = talker.lm_head(states[:, code_start - 1 : code_start - 1 + code_width])

    return torch.nn.functional.cross_entropy(
        predicted.reshape(-1, predicted.size(-1)).float(),
        audio_tokens.reshape(-1),
        ignore_index=-100,
    )


def train_epoch(projection, dataloader, optimizer) -> float:
    projection.train()
    total_loss = 0.0

    progress = tqdm.auto.tqdm(dataloader, desc="Training")
    for batch in progress:
        loss = talker_loss(projection, batch)

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        total_loss += loss.item()
        progress.set_postfix(loss=f"{loss.item():.3f}")

    return total_loss / len(dataloader)

In [ ]:
projection_config = ProjectionConfig(
    input_dim=INPUT_DIM, hidden_dim=HIDDEN_DIM, output_dim=OUTPUT_DIM
)
projection = ProjectionLayer(projection_config).to(DEVICE)
optimizer = torch.optim.AdamW(projection.parameters(), lr=LEARNING_RATE)

print(f"Projection: {INPUT_DIM} -> {HIDDEN_DIM} -> {OUTPUT_DIM}")
print(f"Parameters: {sum(p.numel() for p in projection.parameters()):,}")

# What a checkpoint has to agree with to be resumable: other dimensions are a
# different projection entirely, and a different training set makes the epoch
# count mean something else.
run = {"dimensions": [INPUT_DIM, HIDDEN_DIM, OUTPUT_DIM], "samples": len(dataset)}

epoch_losses = []
first_epoch = 0
if CHECKPOINT_PATH.exists():
    checkpoint = torch.load(CHECKPOINT_PATH, weights_only=True)
    if checkpoint.get("run") == run:
        projection.load_state_dict(checkpoint["projection"])
        optimizer.load_state_dict(checkpoint["optimizer"])
        epoch_losses = checkpoint["losses"]
        first_epoch = len(epoch_losses)
        print(f"Resumed from {CHECKPOINT_PATH} at epoch {first_epoch}")
    else:
        print(f"Ignoring {CHECKPOINT_PATH}: it is from a different run")

for epoch in range(first_epoch, EPOCHS):
    loss = train_epoch(projection, dataloader, optimizer)
    epoch_losses.append(loss)
    torch.save(
        {
            "projection": projection.state_dict(),
            "optimizer": optimizer.state_dict(),
            "losses": epoch_losses,
            "run": run,
        },
        CHECKPOINT_PATH,
    )
    print(f"Epoch {epoch + 1}/{EPOCHS}, Loss: {loss:.4f}")

print(f"Loss: {' -> '.join(f'{loss:.4f}' for loss in epoch_losses)}")
assert epoch_losses[-1] < epoch_losses[0], (
    f"Training did not reduce the loss: {epoch_losses[0]:.4f} -> {epoch_losses[-1]:.4f}"
)

## 11. Check the Projected Path

The training loss is a poor gauge here and it is worth being precise about why.
Teacher forcing hands the talker the ground-truth continuation so far, and a
word costs ~30 tokens: its text, a duration, then ~25 codes. Given the word
already in context, those codes are the talker's own acoustic modelling and owe
nothing to the projection. So the projection is load-bearing on roughly one
position in thirty, and a random one already scores ~2.7 rather than the ~12 a
uniform guess over 157k tokens would -- most of the objective is measuring
something that was never in question.

What settles it is free running: no teacher forcing, the talker reading only
the projection and its own output, exactly as the runtime drives it. If the
words come back, the bridge carries text. This is also the cheapest possible
version of the end-to-end check that `e2e.sh` does with a real ASR model.

In [ ]:
def is_audio_code(token: int) -> bool:
    return AUDIO_CODE_FIRST_TOKEN <= token < AUDIO_CODE_FIRST_TOKEN + AUDIO_CODE_COUNT


def generate_projected(sentence: str) -> list[int]:
    """Free-run the talker from the projection, as the runtime does."""
    projection.eval()
    input_ids = torch.tensor(
        [tokenizer(sentence, add_special_tokens=True)["input_ids"]], device=DEVICE
    )
    hidden_states = extract_hidden_states(input_ids)
    with torch.no_grad():
        projected = projection(hidden_states).to(FROZEN_DTYPE)
        embed = talker.get_input_embeddings()

        def scaffolding(tokens: list[int]) -> torch.Tensor:
            return embed(
                torch.tensor(tokens, dtype=torch.long, device=DEVICE).unsqueeze(0)
            )

        inputs_embeds = torch.cat(
            [
                scaffolding(PROMPT_PREFIX_TOKENS),
                projected,
                scaffolding(PROMPT_SUFFIX_TOKENS),
            ],
            dim=1,
        )
        # Given inputs_embeds and no input_ids, generate returns only what it
        # produced, which is the continuation.
        generated = talker.generate(
            inputs_embeds=inputs_embeds,
            max_new_tokens=MAXIMUM_NEW_TOKENS,
            do_sample=False,
            eos_token_id=END_OF_GENERATION_TOKEN,
            pad_token_id=talker_tokenizer.pad_token_id,
        )
    return generated[0].tolist()


def spoken_words(continuation: list[int]) -> str:
    """The words the talker wrote out, with the codes and markers dropped."""
    text = talker_tokenizer.decode(
        [token for token in continuation if not is_audio_code(token)]
    )
    for marker in ("<|code_start|>", "<|code_end|>", "<|im_end|>"):
        text = text.replace(marker, " ")
    # Durations are <|t_0.52|> and friends; drop them without a regex.
    words = [
        part
        for part in text.split()
        if not (part.startswith("<|t_") and part.endswith("|>"))
    ]
    return " ".join(words)


# Training sentences, so this asks the easier of the two questions: whether the
# projection can encode a sentence it was fitted on at all. Failing here is
# underfitting, not a generalization gap, and no amount of held-out data would
# look better.
for sample in training_samples[:3]:
    sentence = sample["text"]
    continuation = generate_projected(sentence)
    codes = [token for token in continuation if is_audio_code(token)]
    print(f"  in:   {sentence!r}")
    print(f"  said: {spoken_words(continuation)!r}")
    print(
        f"        {len(codes)} codes ({len(codes) / 75:.1f}s), "
        f"stopped on its own: {END_OF_GENERATION_TOKEN in continuation}"
    )

## 12. Export to GGUF

Self-contained GGUF v3 writer (F32 tensors, 32-byte alignment); tensor data
offsets are relative to the aligned start of the tensor-data section.

In [ ]:
GGUF_MAGIC = b"GGUF"
GGUF_VERSION = 3
GGUF_ALIGNMENT = 32
GGUF_TYPE_UINT32 = 4
GGUF_TYPE_STRING = 8
GGML_TYPE_F32 = 0


def align_offset(offset: int) -> int:
    return (offset + GGUF_ALIGNMENT - 1) // GGUF_ALIGNMENT * GGUF_ALIGNMENT


def write_string(f, value: str):
    encoded = value.encode("utf-8")
    f.write(struct.pack("<Q", len(encoded)))
    f.write(encoded)


def save_gguf(projection: ProjectionLayer, output_path: pathlib.Path):
    """Export the projection layer to GGUF for rust_de_llama inference."""
    tensors = {
        name: parameter.detach().cpu().float().numpy()
        for name, parameter in projection.named_parameters()
    }
    metadata = {
        "general.architecture": "projection",
        "general.alignment": GGUF_ALIGNMENT,
        "projection.input_dim": projection.config.input_dim,
        "projection.hidden_dim": projection.config.hidden_dim,
        "projection.output_dim": projection.config.output_dim,
    }

    with open(output_path, "wb") as f:
        f.write(GGUF_MAGIC)
        f.write(struct.pack("<I", GGUF_VERSION))
        f.write(struct.pack("<Q", len(tensors)))
        f.write(struct.pack("<Q", len(metadata)))

        for key, value in metadata.items():
            write_string(f, key)
            if isinstance(value, str):
                f.write(struct.pack("<I", GGUF_TYPE_STRING))
                write_string(f, value)
            else:
                f.write(struct.pack("<I", GGUF_TYPE_UINT32))
                f.write(struct.pack("<I", value))

        offset = 0
        tensor_offsets = {}
        for name, data in tensors.items():
            offset = align_offset(offset)
            tensor_offsets[name] = offset
            offset += data.nbytes

        for name, data in tensors.items():
            write_string(f, name)
            f.write(struct.pack("<I", len(data.shape)))
            # GGUF dimensions are fastest-varying first, the reverse of numpy.
            for dim in reversed(data.shape):
                f.write(struct.pack("<Q", dim))
            f.write(struct.pack("<I", GGML_TYPE_F32))
            f.write(struct.pack("<Q", tensor_offsets[name]))

        f.write(b"\x00" * (align_offset(f.tell()) - f.tell()))
        data_start = f.tell()
        for name, data in tensors.items():
            f.write(b"\x00" * (data_start + tensor_offsets[name] - f.tell()))
            f.write(data.tobytes())

    print(f"Saved GGUF to {output_path}")

In [ ]:
save_gguf(projection, OUTPUT_PATH)

for name, parameter in projection.named_parameters():
    print(
        f"  {name}: {list(parameter.shape)}, {parameter.numel() * 4 / 1024 / 1024:.1f} MB"
    )

## 13. Verify GGUF Round-Trip

Re-parse the exported file with a minimal reader and assert that metadata and
tensors survive byte-exactly.

In [ ]:
def read_gguf(path: pathlib.Path) -> tuple[dict, dict]:
    with open(path, "rb") as f:
        assert f.read(4) == GGUF_MAGIC
        (version,) = struct.unpack("<I", f.read(4))
        assert version == GGUF_VERSION
        (tensor_count,) = struct.unpack("<Q", f.read(8))
        (metadata_count,) = struct.unpack("<Q", f.read(8))

        def read_string() -> str:
            (length,) = struct.unpack("<Q", f.read(8))
            return f.read(length).decode("utf-8")

        metadata = {}
        for _ in range(metadata_count):
            key = read_string()
            (value_type,) = struct.unpack("<I", f.read(4))
            if value_type == GGUF_TYPE_STRING:
                metadata[key] = read_string()
            elif value_type == GGUF_TYPE_UINT32:
                (metadata[key],) = struct.unpack("<I", f.read(4))
            else:
                raise ValueError(f"Unsupported metadata type: {value_type}")

        tensor_infos = []
        for _ in range(tensor_count):
            name = read_string()
            (dimension_count,) = struct.unpack("<I", f.read(4))
            dimensions = [
                struct.unpack("<Q", f.read(8))[0] for _ in range(dimension_count)
            ]
            (dtype,) = struct.unpack("<I", f.read(4))
            (offset,) = struct.unpack("<Q", f.read(8))
            tensor_infos.append((name, dimensions, dtype, offset))

        alignment = metadata.get("general.alignment", GGUF_ALIGNMENT)
        data_start = (f.tell() + alignment - 1) // alignment * alignment
        tensors = {}
        for name, dimensions, dtype, offset in tensor_infos:
            assert dtype == GGML_TYPE_F32
            count = int(numpy.prod(dimensions))
            f.seek(data_start + offset)
            shape = tuple(reversed(dimensions))
            tensors[name] = numpy.frombuffer(
                f.read(count * 4), dtype=numpy.float32
            ).reshape(shape)
        return metadata, tensors


metadata, tensors = read_gguf(OUTPUT_PATH)

assert metadata["general.architecture"] == "projection"
assert metadata["general.alignment"] == GGUF_ALIGNMENT
assert metadata["projection.input_dim"] == projection.config.input_dim
assert metadata["projection.hidden_dim"] == projection.config.hidden_dim
assert metadata["projection.output_dim"] == projection.config.output_dim
for name, parameter in projection.named_parameters():
    numpy.testing.assert_array_equal(
        tensors[name], parameter.detach().cpu().float().numpy()
    )
print(f"GGUF round-trip OK: {sorted(tensors)}")

## 14. Hidden-State Reference for the Runtime

Everything above trains on hidden states this notebook reads from the HF model
in full precision. The runtime reads them from a **quantized GGUF** of the same
model, through llama.cpp and its own tokenizer. Nothing makes those agree, and
two ways they could differ matter:

- **The tokenizers could disagree** on BOS or on the pieces themselves, in which
  case the projection reads positions it was never trained on and no comparison
  of the states is even meaningful. This one is silent — the states would still
  look like states.
- **Quantization moves the states**, which is a matter of degree rather than a
  yes or no.

This writes the reference the Rust side checks both against:
`tests/fixtures/thinker_hidden_states.txt`, holding the token ids and one row of
`hidden_states[-1]` per token. `audio::pipeline::tests::test_thinker_hidden_states_match_the_full_precision_reference`
loads the GGUF, tokenizes the same sentence through the runtime's own
`Tokenizer`, asserts the ids match exactly, and reports the cosine similarity
and relative error per position.

Measured for `SmolLM2-135M-Instruct-Q8_0.gguf`: the ids match (neither end adds
BOS), worst cosine 0.9963, worst relative error 8.6% — and the error grows along
the sentence, 4.4% at the first position to 8.6% by the seventh, as quantization
error accumulates through the causal context. Q8_0 is the mild case; a thinker
at Q4_K_M, which is what the chat models here use, is not measured.

In [ ]:
REFERENCE_PATH = pathlib.Path(
    "../cluster/applications/rust_de_llama/tests/fixtures/thinker_hidden_states.txt"
)
REFERENCE_TEXT = "the quick brown fox jumps over the lazy dog"

# add_special_tokens=True mirrors the runtime, whose Tokenizer::tokenize passes
# add_special = true to llama_tokenize. Both end up adding no BOS here -- the
# GGUF sets tokenizer.ggml.add_bos_token = 0 -- but that is a fact about this
# thinker, not a rule, which is exactly why the ids go in the fixture.
reference_ids = tokenizer(REFERENCE_TEXT, add_special_tokens=True, return_tensors="pt")[
    "input_ids"
]
# fp32 regardless of FROZEN_DTYPE: this is the full-precision side of the
# comparison, so reading it in bf16 would measure this notebook's own rounding
# rather than the runtime's quantization.
reference_model = transformers.AutoModelForCausalLM.from_pretrained(
    THINKER_MODEL, dtype=torch.float32
).eval()
with torch.no_grad():
    reference_states = reference_model(
        input_ids=reference_ids, output_hidden_states=True
    ).hidden_states[-1][0]
del reference_model

REFERENCE_PATH.parent.mkdir(parents=True, exist_ok=True)
REFERENCE_PATH.write_text(
    "\n".join(
        [" ".join(str(token) for token in reference_ids[0].tolist())]
        + [" ".join(f"{value:.6e}" for value in row) for row in reference_states]
    )
    + "\n"
)

print(f"{REFERENCE_TEXT!r}")
print(f"  tokens: {reference_ids[0].tolist()}")
print(f"  pieces: {tokenizer.convert_ids_to_tokens(reference_ids[0].tolist())}")
print(f"  states: {tuple(reference_states.shape)}")
print(f"Wrote {REFERENCE_PATH}")